# NIFTY 50 — Data Sourcing & Validation
### Step 1 of the AlgoChowk Quant Engineer research task

**Objective:** Source a credible, well-documented NIFTY 50 daily OHLC dataset, then validate it
before any event study is built on top of it. Bad data in → meaningless statistics out, so this
step gets treated as seriously as the analysis itself.

**Source:** Yahoo Finance, via the `yfinance` library, ticker `^NSEI` (NSE NIFTY 50 index).
**Date range requested:** 2010-01-01 to 2026-09-01.
**Fields kept:** Date, Open, High, Low, Close (per the assignment's minimum requirement).

**Why Yahoo Finance / yfinance:**
- Free, scriptable, and widely used in retail quant research — reproducible by any reviewer
  without an account or paid subscription.
- Index-level OHLC from Yahoo is not adjusted for dividends/splits the way individual equities
  are, which is actually *appropriate* here since NIFTY is a price index, not a stock.

**Known limitations of this source (documented up front, not discovered later):**
- Yahoo's `^NSEI` history occasionally has short gaps or vendor-side corrections versus the
  official NSE archives — acceptable per the assignment ("a credible, consistently documented
  source is sufficient"), but worth stating rather than hiding.
- Data reflects Yahoo's timezone/session handling; we treat the `Date` as the trading date only
  (no intraday timestamps needed for a daily event study).
- If `yfinance` returns fewer rows than expected for the full 2010–2026 window (e.g. due to a
  rate limit or partial outage), that will be visible in the "actual date coverage" check below —
  don't assume the download silently succeeded.


In [16]:
import yfinance as yf
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

TICKER = "^NSEI"
START = "2010-01-01"
END = "2026-09-01"   # yfinance 'end' is exclusive, so this captures data through 2026-08-31

print(f"Downloading {TICKER} from {START} to {END} ...")
raw = yf.download(TICKER, start=START, end=END, interval="1d", auto_adjust=False, progress=False)
print(f"Raw rows downloaded: {len(raw)}")
raw.tail()

Raw rows downloaded: 4092


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,,
2026-08-25,24334.550781,24334.550781,24334.550781,24115.449219,24175.750000,239500
2026-08-26,24207.750000,24207.750000,24378.599609,24207.750000,24341.949219,244800
2026-08-27,24090.849609,24090.849609,24297.449219,24090.849609,24277.599609,323400
2026-08-28,24175.650391,24175.650391,24188.300781,24076.849609,24122.599609,296100
2026-08-31,24080.400391,24080.400391,24128.699219,23993.599609,24117.550781,710900


In [17]:
# yfinance can return either a flat column index or a MultiIndex (ticker, field)
# depending on version — normalize defensively rather than assuming one shape.
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

raw = raw.reset_index()  # 'Date' becomes a column instead of the index

required_cols = ["Date", "Open", "High", "Low", "Close"]
missing_cols = [c for c in required_cols if c not in raw.columns]
if missing_cols:
    raise ValueError(f"Expected columns missing from download: {missing_cols}. "
                      f"Columns present: {list(raw.columns)}")

df = raw[required_cols].copy()
df["Date"] = pd.to_datetime(df["Date"])
df.head()

Price,Date,Open,High,Low,Close
0,2010-01-04,5200.899902,5238.450195,5167.100098,5232.200195
1,2010-01-05,5277.149902,5288.350098,5242.399902,5277.899902
2,2010-01-06,5278.149902,5310.850098,5260.049805,5281.799805
3,2010-01-07,5281.799805,5302.549805,5244.750000,5263.100098
4,2010-01-08,5264.250000,5276.750000,5234.700195,5244.750000


## Data Validation

Per the assignment's requirements, this checks — and **reports before fixing** — each of:
- Missing / duplicate dates
- Chronological ordering
- Missing or invalid OHLC values
- Suspicious observations (e.g. High < Low, non-positive prices, zero-range days)
- Actual date coverage vs. the requested range

The goal is an audit trail: every row dropped should be visible and justified, not silently
discarded.


In [18]:
validation_log = {}

# --- 1. Missing values ---
na_counts = df[required_cols].isna().sum()
validation_log["missing_values_by_column"] = na_counts.to_dict()
rows_with_na = df[df[required_cols].isna().any(axis=1)]
print("Missing values per column:")
print(na_counts)
print(f"\nRows with at least one NaN: {len(rows_with_na)}")
if len(rows_with_na):
    display(rows_with_na)

Missing values per column:
Price
Date     0
Open     0
High     0
Low      0
Close    0
dtype: int64

Rows with at least one NaN: 0


In [19]:
# --- 2. Duplicate dates ---
dup_mask = df["Date"].duplicated(keep=False)
duplicate_dates = df.loc[dup_mask].sort_values("Date")
validation_log["duplicate_date_rows"] = len(duplicate_dates)
print(f"Duplicate date rows: {len(duplicate_dates)}")
if len(duplicate_dates):
    display(duplicate_dates)

Duplicate date rows: 0


In [20]:
# --- 3. Chronological ordering (checked on the RAW download, before we sort anything) ---
is_sorted_raw = df["Date"].is_monotonic_increasing
validation_log["raw_data_was_chronological"] = bool(is_sorted_raw)
print(f"Raw data already in chronological order: {is_sorted_raw}")

if not is_sorted_raw:
    out_of_order = df[df["Date"].diff().dt.days < 0]
    print(f"Rows where date goes backwards vs. previous row: {len(out_of_order)}")
    display(out_of_order.head(10))

Raw data already in chronological order: True


In [21]:
# --- 4. Invalid / suspicious OHLC relationships ---
# For a valid daily bar: High >= max(Open, Close, Low) and Low <= min(Open, Close, High),
# and all prices should be strictly positive.
non_positive = df[(df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1)]

high_violation = df[df["High"] < df[["Open", "Close", "Low"]].max(axis=1)]
low_violation  = df[df["Low"]  > df[["Open", "Close", "High"]].min(axis=1)]
zero_range     = df[(df["High"] == df["Low"]) & (df["Open"] == df["Close"])]

validation_log.update({
    "non_positive_price_rows": len(non_positive),
    "high_below_required_rows": len(high_violation),
    "low_above_required_rows": len(low_violation),
    "zero_range_days": len(zero_range),
})

print(f"Non-positive prices: {len(non_positive)}")
print(f"High-below-required violations: {len(high_violation)}")
print(f"Low-above-required violations: {len(low_violation)}")
print(f"Zero-range days (High==Low and Open==Close — possible holiday/stale print): {len(zero_range)}")

if len(zero_range):
    display(zero_range.head(10))

Non-positive prices: 0
High-below-required violations: 0
Low-above-required violations: 0
Zero-range days (High==Low and Open==Close — possible holiday/stale print): 0


In [22]:
# --- 5. Suspicious single-day moves (sanity check, not a cleaning rule) ---
# This is diagnostic only — extreme moves are exactly what the research question is about,
# so they should NOT be dropped as "outliers". This just flags candidates for a manual look
# (e.g. a fat-fingered print vs. a genuine crash day like 2020-03-23).
df_sorted_check = df.sort_values("Date")
daily_ret_check = df_sorted_check["Close"].pct_change()
extreme_days = df_sorted_check.loc[daily_ret_check.abs() > 0.10, ["Date", "Open", "High", "Low", "Close"]]
extreme_days = extreme_days.assign(pct_change=daily_ret_check[daily_ret_check.abs() > 0.10].values)

print(f"Days with |daily return| > 10%: {len(extreme_days)}")
display(extreme_days)
# These are kept in the dataset — they're candidate EVENTS for the study, not data errors —
# unless a specific row is independently confirmed to be a vendor error.

Days with |daily return| > 10%: 1


Price,Date,Open,High,Low,Close,pct_change
2499,2020-03-23,7945.700195,8159.25,7583.600098,7610.25,-0.129805


In [23]:
# --- 6. Actual date coverage vs. requested range ---
coverage = {
    "requested_start": START,
    "requested_end": END,
    "actual_start": str(df["Date"].min().date()),
    "actual_end": str(df["Date"].max().date()),
    "row_count": len(df),
}
validation_log["coverage"] = coverage
print("Requested vs. actual coverage:")
for k, v in coverage.items():
    print(f"  {k}: {v}")

# Rough expected trading-day count sanity check (NSE trades ~248-250 days/year historically).
years_span = (df["Date"].max() - df["Date"].min()).days / 365.25
approx_expected = years_span * 248
print(f"\nApprox. expected trading days over this span (~248/yr heuristic): {approx_expected:.0f}")
print(f"Actual rows: {len(df)}  (large gap here would flag a partial/failed download)")

Requested vs. actual coverage:
  requested_start: 2010-01-01
  requested_end: 2026-09-01
  actual_start: 2010-01-04
  actual_end: 2026-08-31
  row_count: 4092

Approx. expected trading days over this span (~248/yr heuristic): 4130
Actual rows: 4092  (large gap here would flag a partial/failed download)


## Cleaning

Based on the checks above, apply cleaning **decisions**, not blanket defaults. Each decision is
stated explicitly so a reviewer can see (and disagree with) the reasoning:

1. **Drop rows with any NaN in Date/Open/High/Low/Close** — an incomplete bar can't be used to
   compute a return, and silently forward-filling a price would fabricate a data point.
2. **Drop duplicate dates, keeping the first occurrence** — only relevant if step above found any.
3. **Sort strictly chronologically by Date** and verify the result is monotonic increasing.
4. **Do not drop extreme-return days** — those are the subject of the study, not noise.
5. **Flag (but don't auto-drop) OHLC-relationship violations** — if any exist, they're inspected
   manually below rather than removed automatically, since a single bad row is a very different
   problem from a systematic vendor issue.


In [24]:
rows_before = len(df)

# 1. Drop incomplete rows
df_clean = df.dropna(subset=required_cols).copy()
dropped_na = rows_before - len(df_clean)

# 2. Drop duplicate dates (keep first)
rows_before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="Date", keep="first")
dropped_dupes = rows_before_dedup - len(df_clean)

# 3. Sort chronologically and reset index
df_clean = df_clean.sort_values("Date").reset_index(drop=True)
is_sorted_final = df_clean["Date"].is_monotonic_increasing

print(f"Rows dropped for missing values: {dropped_na}")
print(f"Rows dropped for duplicate dates: {dropped_dupes}")
print(f"Final row count: {len(df_clean)} (started at {rows_before})")
print(f"Final data is chronologically sorted: {is_sorted_final}")

assert is_sorted_final, "Post-cleaning data must be strictly chronological before any event study."
assert df_clean[required_cols].isna().sum().sum() == 0, "NaNs remain after cleaning."
assert df_clean["Date"].is_unique, "Duplicate dates remain after cleaning."

validation_log["dropped_na_rows"] = int(dropped_na)
validation_log["dropped_duplicate_rows"] = int(dropped_dupes)
validation_log["final_row_count"] = int(len(df_clean))

df_clean.head()

Rows dropped for missing values: 0
Rows dropped for duplicate dates: 0
Final row count: 4092 (started at 4092)
Final data is chronologically sorted: True


Price,Date,Open,High,Low,Close
0,2010-01-04,5200.899902,5238.450195,5167.100098,5232.200195
1,2010-01-05,5277.149902,5288.350098,5242.399902,5277.899902
2,2010-01-06,5278.149902,5310.850098,5260.049805,5281.799805
3,2010-01-07,5281.799805,5302.549805,5244.750000,5263.100098
4,2010-01-08,5264.250000,5276.750000,5234.700195,5244.750000


In [25]:
# Manual inspection point for any flagged OHLC violations (only runs if any exist)
flagged = pd.concat([high_violation, low_violation, non_positive]).drop_duplicates(subset="Date")
if len(flagged):
    print(f"{len(flagged)} row(s) flagged for manual review — inspect before treating as valid:")
    display(flagged)
else:
    print("No OHLC-relationship violations found — nothing flagged for manual review.")

No OHLC-relationship violations found — nothing flagged for manual review.


In [26]:
import json as _json

print(_json.dumps(validation_log, indent=2, default=str))

{
  "missing_values_by_column": {
    "Date": 0,
    "Open": 0,
    "High": 0,
    "Low": 0,
    "Close": 0
  },
  "duplicate_date_rows": 0,
  "raw_data_was_chronological": true,
  "non_positive_price_rows": 0,
  "high_below_required_rows": 0,
  "low_above_required_rows": 0,
  "zero_range_days": 0,
  "coverage": {
    "requested_start": "2010-01-01",
    "requested_end": "2026-09-01",
    "actual_start": "2010-01-04",
    "actual_end": "2026-08-31",
    "row_count": 4092
  },
  "dropped_na_rows": 0,
  "dropped_duplicate_rows": 0,
  "final_row_count": 4092
}


In [27]:
OUTPUT_PATH = "nifty50_daily_clean_2010_2026.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to: {OUTPUT_PATH}")
print(f"Shape: {df_clean.shape}")
df_clean.describe()

Saved cleaned dataset to: nifty50_daily_clean_2010_2026.csv
Shape: (4092, 5)


Price,Date,Open,High,Low,Close
count,4092,4092.000000,4092.000000,4092.000000,4092.000000
mean,2018-05-08 10:34:29.208211200,12343.788039,12402.645062,12268.322503,12336.661885
min,2010-01-04 00:00:00,4623.149902,4623.149902,4531.149902,4544.200195
25%,2014-03-04 18:00:00,6352.212646,6395.512573,6333.875122,6359.337402
50%,2018-05-14 12:00:00,10363.600098,10420.174805,10316.649902,10374.325195
75%,2022-07-07 06:00:00,17540.975098,17638.886719,17430.787109,17540.287109
max,2026-08-31 00:00:00,26333.699219,26373.199219,26210.050781,26328.550781
std,NaN,6580.913160,6605.024205,6552.245718,6579.458237


In [31]:
import pandas as pd
import numpy as np

# 1. Reload the clean data and rebuild the events (Self-Contained)
df_clean = pd.read_csv("nifty50_daily_clean_2010_2026.csv", parse_dates=["Date"])
df_clean = df_clean.sort_values("Date").reset_index(drop=True)

FALL_THRESHOLD = -0.02
K = 3  # 3-day holding period

df_clean["daily_return"] = df_clean["Close"].pct_change()
df_clean["is_event"] = df_clean["daily_return"] <= FALL_THRESHOLD
df_clean["entry_price_t1_open"] = df_clean["Open"].shift(-1)
df_clean[f"fwd_return_{K}d"] = df_clean["Close"].shift(-K) / df_clean["entry_price_t1_open"] - 1

return_cols = [f"fwd_return_{K}d"]
events = (df_clean.loc[df_clean["is_event"], ["Date"] + return_cols]
          .dropna(subset=return_cols)
          .sort_values("Date")
          .reset_index(drop=True))

# 2. Chronological Split (In-Sample vs. Out-of-Sample)
in_sample = events[events["Date"].dt.year <= 2019].copy()
out_sample = events[events["Date"].dt.year >= 2020].copy()

def period_stats(sub, k=3):
    col = f"fwd_return_{k}d"
    r = sub[col]
    return {
        "n_events": len(r),
        "mean_return_3d": r.mean(),
        "win_rate_3d": (r > 0).mean(),
    }

split_summary = pd.DataFrame({
    "In-Sample (2010-2019)": period_stats(in_sample),
    "Out-of-Sample (2020-2026)": period_stats(out_sample),
}).T
split_summary.index.name = "Period"

print("--- Out-of-Sample Validation ---")
split_display = split_summary.copy()
split_display["mean_return_3d"] = split_display["mean_return_3d"].map(lambda x: f"{x:.2%}")
split_display["win_rate_3d"] = split_display["win_rate_3d"].map(lambda x: f"{x:.2%}")
display(split_display)

# 3. Simple Vectorized Backtest (3-Day Holding Period)
TRANSACTION_COST = 0.0010   # 10 bps round-trip cost

trades = events[["Date", f"fwd_return_{K}d"]].copy()
trades.rename(columns={f"fwd_return_{K}d": "gross_return"}, inplace=True)

# Apply transaction friction
trades["net_return"] = trades["gross_return"] - TRANSACTION_COST

# Build equity curve assuming full compounding per trade
trades["equity_curve"] = (1 + trades["net_return"]).cumprod()

total_trades = len(trades)
cumulative_return = trades["equity_curve"].iloc[-1] - 1

# Calculate Maximum Drawdown
running_max = trades["equity_curve"].cummax()
drawdown = trades["equity_curve"] / running_max - 1
max_drawdown = drawdown.min()

print("\n--- Event-Driven Backtest (After 10 bps Cost) ---")
print(f"Total Number of Trades: {total_trades}")
print(f"Cumulative Return:      {cumulative_return:.2%}")
print(f"Maximum Drawdown:       {max_drawdown:.2%}")

--- Out-of-Sample Validation ---


,n_events,mean_return_3d,win_rate_3d
Period,,,
In-Sample (2010-2019),59.0,0.45%,61.02%
Out-of-Sample (2020-2026),49.0,0.14%,59.18%



--- Event-Driven Backtest (After 10 bps Cost) ---
Total Number of Trades: 108
Cumulative Return:      18.57%
Maximum Drawdown:       -39.10%


In [32]:
import pandas as pd
import numpy as np

# 1. Setup & Calculate Returns
K = 3
df = pd.read_csv("nifty50_daily_clean_2010_2026.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
df["daily_return"] = df["Close"].pct_change()
df["is_event"] = df["daily_return"] <= -0.02
df["entry_price"] = df["Open"].shift(-1)
df["fwd_return_3d"] = df["Close"].shift(-K) / df["entry_price"] - 1

# 2. Strict Non-Overlapping Filter
valid_idx = []
last_exit = -1
for idx in df[df["is_event"]].index:
    if idx > last_exit:
        valid_idx.append(idx)
        last_exit = idx + K

events = df.loc[valid_idx].dropna(subset=["fwd_return_3d"]).copy()
baseline = df.dropna(subset=["fwd_return_3d"]).copy()

in_sample = events[events["Date"].dt.year <= 2019]
out_sample = events[events["Date"].dt.year >= 2020]

# 3. Master Table Generator
def stats(series):
    if len(series) == 0: return ["N/A"] * 6
    return [
        len(series), f"{series.mean():.2%}", f"{series.median():.2%}",
        f"{(series > 0).mean():.2%}", f"{series.std():.2%}",
        f"{series.min():.2%} / {series.max():.2%}"
    ]

table = pd.DataFrame({
    "Full Sample": stats(events["fwd_return_3d"]),
    "In-Sample": stats(in_sample["fwd_return_3d"]),
    "Out-Sample": stats(out_sample["fwd_return_3d"]),
    "Baseline": stats(baseline["fwd_return_3d"])
}, index=["N", "Mean Return", "Median Return", "Win Rate", "Std Dev", "Min/Max"]).T

print("--- MASTER RESULTS TABLE ---")
display(table)

# 4. Non-Parametric Permutation Test (10,000 samples)
np.random.seed(42)
obs_diff = events["fwd_return_3d"].mean() - baseline["fwd_return_3d"].mean()
combined = np.concatenate([events["fwd_return_3d"].values, baseline["fwd_return_3d"].values])
count = 0
n_perms = 10000

for _ in range(n_perms):
    np.random.shuffle(combined)
    sim_diff = combined[:len(events)].mean() - combined[len(events):].mean()
    if sim_diff >= obs_diff: count += 1

print(f"\nPermutation Test p-value: {count / n_perms:.4f}")

# 5. Non-Overlapping Backtest (10bps & 20bps)
print("\n--- VALIDATED BACKTEST ---")
for cost in [0.0010, 0.0020]:
    eq = (1 + (events["fwd_return_3d"] - cost)).cumprod()
    cum_ret = eq.iloc[-1] - 1
    max_dd = ((eq / eq.cummax()) - 1).min()
    print(f"{int(cost*10000)} bps Slippage -> Cum Ret: {cum_ret:.2%} | Max DD: {max_dd:.2%}")

--- MASTER RESULTS TABLE ---


,N,Mean Return,Median Return,Win Rate,Std Dev,Min/Max
Full Sample,89,0.34%,0.55%,60.67%,3.04%,-11.01% / 10.11%
In-Sample,52,0.43%,0.46%,59.62%,2.23%,-4.73% / 6.00%
Out-Sample,37,0.21%,0.65%,62.16%,3.95%,-11.01% / 10.11%
Baseline,4089,0.02%,0.09%,52.43%,1.68%,-11.67% / 11.96%



Permutation Test p-value: 0.0414

--- VALIDATED BACKTEST ---
10 bps Slippage -> Cum Ret: 18.57% | Max DD: -26.28%
20 bps Slippage -> Cum Ret: 8.48% | Max DD: -28.27%
